# Brain Tumor Segmentation - U-Net Training

**Architecture:** Lightweight U-Net with MobileNetV2 Encoder + ASPP Bridge

**Dataset:** BRISC 2025 Segmentation Task (5000 train / 1000 test)

**Pipeline:** Train segmentation model -> Evaluate -> Feed segmented images to NeuroFusionNet classifier

---

## 1. Setup & Imports

In [ ]:
import os, sys, json, time, random
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from PIL import Image
import cv2
from tqdm.notebook import tqdm

# Add project root — go up 2 levels from combination_of_segmentation_CNN
# combination_of_segmentation_CNN -> neurofusionnet -> Final Year Project
NOTEBOOK_DIR = Path(os.getcwd())
if 'combination_of_segmentation_CNN' in str(NOTEBOOK_DIR):
    PROJECT_ROOT = NOTEBOOK_DIR.parent.parent
elif 'neurofusionnet' in str(NOTEBOOK_DIR):
    PROJECT_ROOT = NOTEBOOK_DIR.parent
else:
    PROJECT_ROOT = NOTEBOOK_DIR

sys.path.insert(0, str(PROJECT_ROOT))
print(f'Project root: {PROJECT_ROOT}')

from neurofusionnet.combination_of_segmentation_CNN.segmentation_model import (
    LightweightUNet, create_unet, CombinedSegLoss, DiceLoss, dice_score, iou_score
)
from neurofusionnet.combination_of_segmentation_CNN.segmentation_dataset import (
    BRISCSegDataset, JointTransform, get_seg_dataloaders
)

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Paths
DATA_DIR = str(PROJECT_ROOT / 'brisc2025' / 'segmentation_task')
CHECKPOINT_DIR = str(PROJECT_ROOT / 'checkpoints')
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f'Data: {DATA_DIR}')
print(f'Checkpoints: {CHECKPOINT_DIR}')

## 2. Data Exploration

In [ ]:
# Load raw dataset (no augmentation) for exploration
raw_dataset = BRISCSegDataset(DATA_DIR, split='train')
print(f'Total training samples: {len(raw_dataset)}')

# Tumor type distribution
tumor_types = [raw_dataset.get_tumor_type(i) for i in range(len(raw_dataset))]
from collections import Counter
dist = Counter(tumor_types)
print(f'\nTumor type distribution:')
for t, c in sorted(dist.items()):
    print(f'  {t}: {c} ({c/len(raw_dataset)*100:.1f}%)')

In [ ]:
# Visualize sample image-mask pairs
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
fig.suptitle('Sample MRI Images and Segmentation Masks', fontsize=16, fontweight='bold')

indices = random.sample(range(len(raw_dataset)), 6)
for i, idx in enumerate(indices):
    img, mask = raw_dataset[idx]
    tumor_type = raw_dataset.get_tumor_type(idx)
    row = i // 2
    col = (i % 2) * 2
    
    axes[row, col].imshow(img.permute(1,2,0).numpy())
    axes[row, col].set_title(f'{tumor_type} - MRI', fontweight='bold')
    axes[row, col].axis('off')
    
    axes[row, col+1].imshow(mask.squeeze().numpy(), cmap='hot')
    coverage = mask.sum().item() / mask.numel() * 100
    axes[row, col+1].set_title(f'Mask ({coverage:.1f}% coverage)')
    axes[row, col+1].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Mask coverage statistics
coverages = []
for i in range(min(500, len(raw_dataset))):
    _, mask = raw_dataset[i]
    coverages.append(mask.sum().item() / mask.numel() * 100)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.hist(coverages, bins=30, color='#6366f1', edgecolor='white', alpha=0.8)
ax1.set_xlabel('Tumor Coverage (%)')
ax1.set_ylabel('Count')
ax1.set_title('Distribution of Tumor Area Coverage')

ax2.bar(dist.keys(), dist.values(), color=['#ef4444', '#f59e0b', '#3b82f6', '#22c55e'][:len(dist)])
ax2.set_title('Tumor Type Distribution')
ax2.set_ylabel('Count')
plt.tight_layout()
plt.show()

print(f'Mean coverage: {np.mean(coverages):.2f}%')
print(f'Median coverage: {np.median(coverages):.2f}%')
print(f'Max coverage: {np.max(coverages):.2f}%')

## 3. DataLoaders with Augmentation

In [ ]:
# Hyperparameters
IMG_SIZE = 224
BATCH_SIZE = 16
NUM_WORKERS = 2
EPOCHS = 30
LR = 1e-4
PATIENCE = 7

# Create dataloaders
train_loader, val_loader, test_loader = get_seg_dataloaders(
    DATA_DIR, batch_size=BATCH_SIZE, val_split=0.2,
    img_size=IMG_SIZE, num_workers=NUM_WORKERS
)

print(f'Train batches: {len(train_loader)} ({len(train_loader)*BATCH_SIZE} images)')
print(f'Val batches:   {len(val_loader)}')
print(f'Test batches:  {len(test_loader)}')

# Verify batch
imgs, masks = next(iter(train_loader))
print(f'\nBatch shapes - Images: {imgs.shape}, Masks: {masks.shape}')
print(f'Image range: [{imgs.min():.2f}, {imgs.max():.2f}]')
print(f'Mask unique values: {masks.unique().tolist()}')

## 4. Model Architecture

In [ ]:
# Create U-Net
model = create_unet(pretrained=True, freeze_encoder=0.5).to(device)

# Parameter count
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params = total_params - trainable_params

print('=' * 50)
print('  Lightweight U-Net Architecture')
print('=' * 50)
print(f'  Encoder:    MobileNetV2 (pretrained)')
print(f'  Bridge:     ASPP (multi-scale context)')
print(f'  Decoder:    4 upsampling blocks + skip connections')
print(f'  Output:     Binary mask (sigmoid)')
print('-' * 50)
print(f'  Total params:     {total_params:>10,} ({total_params/1e6:.2f}M)')
print(f'  Trainable:        {trainable_params:>10,} ({trainable_params/1e6:.2f}M)')
print(f'  Frozen:           {frozen_params:>10,} ({frozen_params/1e6:.2f}M)')
print('=' * 50)

# Test forward pass
with torch.no_grad():
    test_input = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(device)
    test_output = model(test_input)
    print(f'\nForward pass: {test_input.shape} -> {test_output.shape}')

## 5. Training

In [ ]:
# Loss, optimizer, scheduler
criterion = CombinedSegLoss(dice_weight=0.5, bce_weight=0.5)
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR, weight_decay=1e-5
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

# Mixed precision
use_amp = device.type == 'cuda'
scaler = torch.amp.GradScaler('cuda') if use_amp else None

print(f'Loss: CombinedSegLoss (0.5*Dice + 0.5*BCE)')
print(f'Optimizer: Adam (lr={LR}, wd=1e-5)')
print(f'Scheduler: CosineAnnealingLR (T_max={EPOCHS})')
print(f'AMP: {use_amp}')

In [ ]:
# Training loop
history = {
    'train_loss': [], 'val_loss': [],
    'train_dice': [], 'val_dice': [],
    'train_iou': [], 'val_iou': [],
    'lr': []
}

best_dice = 0.0
no_improve = 0

print(f'Starting training for {EPOCHS} epochs...')
print(f'{"Epoch":>6} | {"Train Loss":>10} | {"Val Loss":>10} | {"Train Dice":>10} | {"Val Dice":>10} | {"Val IoU":>10} | {"LR":>10}')
print('-' * 85)

for epoch in range(1, EPOCHS + 1):
    start = time.time()
    
    # --- Train ---
    model.train()
    train_loss, train_dice_sum, train_iou_sum = 0, 0, 0
    
    for images, masks in tqdm(train_loader, desc=f'Epoch {epoch}', leave=False):
        images, masks = images.to(device), masks.to(device)
        optimizer.zero_grad()
        
        if use_amp:
            with torch.amp.autocast('cuda'):
                preds = model(images)
                loss = criterion(preds, masks)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            preds = model(images)
            loss = criterion(preds, masks)
            loss.backward()
            optimizer.step()
        
        train_loss += loss.item()
        with torch.no_grad():
            train_dice_sum += dice_score(preds, masks).item()
            train_iou_sum += iou_score(preds, masks).item()
    
    n_train = len(train_loader)
    avg_train_loss = train_loss / n_train
    avg_train_dice = train_dice_sum / n_train
    avg_train_iou = train_iou_sum / n_train
    
    # --- Validate ---
    model.eval()
    val_loss, val_dice_sum, val_iou_sum = 0, 0, 0
    with torch.no_grad():
        for images, masks in val_loader:
            images, masks = images.to(device), masks.to(device)
            preds = model(images)
            loss = criterion(preds, masks)
            val_loss += loss.item()
            val_dice_sum += dice_score(preds, masks).item()
            val_iou_sum += iou_score(preds, masks).item()
    
    n_val = max(len(val_loader), 1)
    avg_val_loss = val_loss / n_val
    avg_val_dice = val_dice_sum / n_val
    avg_val_iou = val_iou_sum / n_val
    
    scheduler.step()
    current_lr = scheduler.get_last_lr()[0]
    
    # Log
    history['train_loss'].append(avg_train_loss)
    history['val_loss'].append(avg_val_loss)
    history['train_dice'].append(avg_train_dice)
    history['val_dice'].append(avg_val_dice)
    history['train_iou'].append(avg_train_iou)
    history['val_iou'].append(avg_val_iou)
    history['lr'].append(current_lr)
    
    # Best model
    tag = ''
    if avg_val_dice > best_dice:
        best_dice = avg_val_dice
        no_improve = 0
        tag = ' << BEST'
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'dice_score': best_dice,
        }, os.path.join(CHECKPOINT_DIR, 'unet_segmentation.pth'))
    else:
        no_improve += 1
    
    elapsed = time.time() - start
    print(f'{epoch:>6} | {avg_train_loss:>10.4f} | {avg_val_loss:>10.4f} | {avg_train_dice:>10.4f} | {avg_val_dice:>10.4f} | {avg_val_iou:>10.4f} | {current_lr:>10.2e} | {elapsed:.0f}s{tag}')
    
    # Early stopping
    if no_improve >= PATIENCE:
        print(f'\nEarly stopping at epoch {epoch} (no improvement for {PATIENCE} epochs)')
        break

# Save history
with open(os.path.join(CHECKPOINT_DIR, 'seg_training_history.json'), 'w') as f:
    json.dump(history, f, indent=2)

print(f'\nTraining complete! Best Val Dice: {best_dice:.4f}')
print(f'Model saved to: {os.path.join(CHECKPOINT_DIR, "unet_segmentation.pth")}')

## 6. Training Curves

In [ ]:
epochs_range = range(1, len(history['train_loss']) + 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss
axes[0].plot(epochs_range, history['train_loss'], 'b-', linewidth=2, label='Train')
axes[0].plot(epochs_range, history['val_loss'], 'r-', linewidth=2, label='Validation')
axes[0].set_title('Loss (Dice + BCE)', fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Dice
axes[1].plot(epochs_range, history['train_dice'], 'b-', linewidth=2, label='Train')
axes[1].plot(epochs_range, history['val_dice'], 'g-', linewidth=2, label='Validation')
axes[1].set_title('Dice Score', fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Dice')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

# IoU
axes[2].plot(epochs_range, history['train_iou'], 'b-', linewidth=2, label='Train')
axes[2].plot(epochs_range, history['val_iou'], 'm-', linewidth=2, label='Validation')
axes[2].set_title('IoU Score', fontweight='bold')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('IoU')
axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.suptitle('Segmentation Training Curves', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 7. Test Set Evaluation

In [ ]:
# Load best model
best_ckpt = os.path.join(CHECKPOINT_DIR, 'unet_segmentation.pth')
if os.path.exists(best_ckpt):
    ckpt = torch.load(best_ckpt, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    print(f'Loaded best model (epoch {ckpt["epoch"]}, dice={ckpt["dice_score"]:.4f})')

model.eval()
test_dice_all, test_iou_all = [], []

with torch.no_grad():
    for images, masks in tqdm(test_loader, desc='Testing'):
        images, masks = images.to(device), masks.to(device)
        preds = model(images)
        test_dice_all.append(dice_score(preds, masks).item())
        test_iou_all.append(iou_score(preds, masks).item())

print(f'\nTest Set Results:')
print(f'  Dice Score: {np.mean(test_dice_all):.4f} +/- {np.std(test_dice_all):.4f}')
print(f'  IoU Score:  {np.mean(test_iou_all):.4f} +/- {np.std(test_iou_all):.4f}')

## 8. Visual Results - Predictions vs Ground Truth

In [ ]:
# Visualize predictions on test images
model.eval()
test_dataset = BRISCSegDataset(DATA_DIR, split='test',
                                transform=JointTransform(img_size=IMG_SIZE, is_train=False))

fig, axes = plt.subplots(4, 4, figsize=(16, 16))
fig.suptitle('Segmentation Results: Original | Ground Truth | Prediction | Overlay', fontsize=14, fontweight='bold')

mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])

indices = random.sample(range(len(test_dataset)), 4)
for row, idx in enumerate(indices):
    img, mask_gt = test_dataset[idx]
    
    # Predict
    with torch.no_grad():
        pred_logits = model(img.unsqueeze(0).to(device))
        pred_mask = (torch.sigmoid(pred_logits) > 0.5).float().squeeze().cpu().numpy()
    
    # Denormalize image
    img_np = img.numpy().transpose(1, 2, 0)
    img_np = np.clip(img_np * std + mean, 0, 1)
    mask_gt_np = mask_gt.squeeze().numpy()
    
    # Original
    axes[row, 0].imshow(img_np)
    axes[row, 0].set_title('Original MRI')
    axes[row, 0].axis('off')
    
    # Ground truth
    axes[row, 1].imshow(mask_gt_np, cmap='hot')
    axes[row, 1].set_title('Ground Truth')
    axes[row, 1].axis('off')
    
    # Prediction
    axes[row, 2].imshow(pred_mask, cmap='hot')
    d = dice_score(pred_logits.cpu(), mask_gt.unsqueeze(0)).item()
    axes[row, 2].set_title(f'Prediction (Dice: {d:.3f})')
    axes[row, 2].axis('off')
    
    # Overlay
    overlay = img_np.copy()
    overlay[pred_mask > 0.5] = overlay[pred_mask > 0.5] * 0.5 + np.array([1, 0.2, 0.2]) * 0.5
    axes[row, 3].imshow(overlay)
    axes[row, 3].set_title('Overlay')
    axes[row, 3].axis('off')

plt.tight_layout()
plt.show()

## 9. Research Analysis - Segmentation Impact on Classification

### Approach Comparison for Healthcare Deployment

| Approach | Description | Pros | Cons | Recommendation |
|----------|-------------|------|------|----------------|
| **A: Baseline** | Raw image -> Classifier | Simple, no seg errors | Background noise affects classifier | Baseline only |
| **B: Seg-Guided (Ours)** | U-Net mask -> Masked image -> Classifier | Focuses on tumor ROI, reduces false features | Segmentation errors can propagate | **Recommended** |
| **C: Multi-Task** | Joint seg + classification training | End-to-end, shared representations | Complex training, harder to debug | Research |
| **D: Aux Features** | Seg features concatenated as auxiliary input | Richer representation | More parameters, overfitting risk | Advanced |

### Healthcare Impact

**Segmentation-guided classification is recommended** for healthcare because:

1. **Explainability**: The segmentation mask shows *where* the model sees the tumor - critical for radiologist trust
2. **Noise reduction**: Removing background tissue reduces false positive features
3. **Tumor localization**: Provides spatial information (location, size) beyond just classification
4. **Modularity**: Each component can be independently validated and improved
5. **Clinical workflow**: Matches how radiologists work - first find the tumor, then characterize it

## 10. Generate Segmented Images for Classifier

In [ ]:
# Show pipeline: Original -> Mask -> Masked Image
from neurofusionnet.combination_of_segmentation_CNN.pipeline import SegClassPipeline

pipeline = SegClassPipeline(
    seg_checkpoint=os.path.join(CHECKPOINT_DIR, 'unet_segmentation.pth'),
    cls_checkpoint=os.path.join(CHECKPOINT_DIR, 'best_model.pth'),
)

# Run on sample images
test_images_dir = Path(DATA_DIR) / 'test' / 'images'
sample_files = list(test_images_dir.glob('*.jpg'))[:4]

fig, axes = plt.subplots(len(sample_files), 4, figsize=(16, 4*len(sample_files)))
fig.suptitle('Full Pipeline: MRI -> Segmentation -> Mask -> Classification', fontsize=14, fontweight='bold')

for i, img_path in enumerate(sample_files):
    image = Image.open(img_path).convert('RGB')
    results = pipeline.full_pipeline(image)
    
    # Original
    axes[i, 0].imshow(np.array(image.resize((224, 224))))
    axes[i, 0].set_title('Original MRI')
    axes[i, 0].axis('off')
    
    # Probability map
    axes[i, 1].imshow(results['prob_map'], cmap='magma')
    axes[i, 1].set_title(f'Seg Map ({results["tumor_percentage"]:.1f}%)')
    axes[i, 1].axis('off')
    
    # Masked image
    axes[i, 2].imshow(np.array(results['masked_image']))
    axes[i, 2].set_title('Masked ROI')
    axes[i, 2].axis('off')
    
    # Classification result
    axes[i, 3].bar(results['all_probs'].keys(), results['all_probs'].values(),
                   color=['#ef4444', '#f59e0b', '#3b82f6', '#22c55e'])
    axes[i, 3].set_title(f'Pred: {results["prediction"]} ({results["confidence"]:.1%})')
    axes[i, 3].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 11. Summary

### Results
- **U-Net Model**: ~5.2M parameters (lightweight, deployable)
- **Loss**: Combined Dice + BCE for balanced training
- **Training**: CosineAnnealingLR + AMP + Early Stopping

### Next Steps
1. **Fine-tune classifier** on segmented images (`pipeline.finetune_classifier_on_segmented()`)
2. **Run full app**: `streamlit run app.py` for the interactive dashboard
3. **Enable AI reports**: Set `GEMINI_API_KEY` for free LLM-powered clinical reports